# 🎬 Montana State University - M090 Simple Slide Video Engine (Google Colab GPU Edition)

**Google의 무료 T4 GPU**를 활용하여 Montana State University M090 강의 슬라이드와 프리젠터 대본을 고화질(1080p) MP4 영상으로 초고속 렌더링하는 전용 Colab 노트북입니다.

---
### 🌟 주요 특징
1. **무료 T4 GPU 하드웨어 가속 (`h264_nvenc`)**: 로컬 CPU 대비 수십 배 빠른 렌더링 속도
2. **내 PC 자원 소모 제로**: 클라우드 인스턴스에서 모든 영상 합성 및 오디오 생성 완료
3. **PresenterMode 대본 완벽 연동**: Prof. Eunju Park & TA Sora의 2인 발화 및 수학 공식 발음 필터 적용
4. **하단 글래스모피즘 HUD 자막 바**: 발화자 네온 링 배지와 읽기 쉬운 자막 자동 오버레이
5. **구글 드라이브 원클릭 저장**: 완성된 영상이 내 구글 드라이브로 즉시 자동 백업

## 1단계: 무료 GPU 확인 및 필수 패키지 설치
상단 메뉴의 **[런타임] → [런타임 유형 변경]**에서 하드웨어 가속기가 **T4 GPU**로 설정되어 있는지 확인 후 실행하세요.

In [ ]:
# GPU 가속기 상태 점검
!nvidia-smi

# 필수 라이브러리 및 헤드리스 브라우저 설치
!pip install -q edge-tts imageio-ffmpeg pillow playwright
!playwright install chromium
!playwright install-deps

## 2단계: 프로젝트 코드 준비 (GitHub 클론)
GitHub 레포지토리를 Colab 가상 머신에 복제합니다.

In [ ]:
import os
%cd /content
if not os.path.exists('/content/montana-lecture-app'):
    !git clone https://github.com/marylatte11-svg/montana-lecture-app.git /content/montana-lecture-app
%cd /content/montana-lecture-app
!pwd

## 3단계 (선택): 구글 드라이브 마운트
완성된 강의 영상을 내 구글 드라이브에 바로 저장하려면 아래 코드를 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 드라이브 저장 대상 폴더 생성 (예: 'My Drive/MSU_M090_Videos')
DRIVE_OUT = '/content/drive/MyDrive/MSU_M090_Videos'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f"✅ Google Drive 저장 위치: {DRIVE_OUT}")

## 4단계: 제2의 영상 엔진(Simple Slide Video Engine) 실행
`--lecture` 인자에 원하는 강의 번호(1~45)를 입력하여 실행합니다.
NVIDIA T4 GPU의 `h264_nvenc` 인코더가 자동으로 감지되어 단 수십 초 만에 풀 마스터 영상이 완성됩니다!

In [ ]:
# Lecture 01 렌더링 실행 (1~45번 중 원하는 강의 번호로 변경 가능)
LECTURE_ID = 1
!python scripts/simple_video_engine/render_simple_video.py --lecture {LECTURE_ID}

## 5단계: 완성된 마스터 영상 확인 및 구글 드라이브 복사 / 다운로드

In [ ]:
import shutil
import glob
from google.colab import files

# 렌더링된 마스터 영상 검색
master_videos = glob.glob(f"/content/montana-lecture-app/Montana_State_Univ/simple_videos/Lecture{LECTURE_ID:02d}/*_Simple_Master.mp4")

if master_videos:
    target_video = master_videos[0]
    size_mb = os.path.getsize(target_video) / (1024 * 1024)
    print(f"🎬 완성된 비디오: {os.path.basename(target_video)} ({size_mb:.2f} MB)")
    
    # 1. 구글 드라이브로 복사
    if os.path.exists('/content/drive/MyDrive'):
        dest_drive = os.path.join(DRIVE_OUT, os.path.basename(target_video))
        shutil.copy2(target_video, dest_drive)
        print(f"☁️ 구글 드라이브 저장 완료: {dest_drive}")
    
    # 2. 로컬 브라우저로 직접 다운로드
    print("⬇️ 브라우저 다운로드를 시작합니다...")
    files.download(target_video)
else:
    print("❌ 비디오 파일을 찾을 수 없습니다. 4단계를 먼저 실행해 주세요.")